### Preparation

In [ ]:
import torch
import numpy as np

from tqdm import tqdm

from layers import *
from basicts.metrics import masked_mae, masked_mape, masked_rmse
from build_dataset import construct_data


#### Parameter Settings

In [ ]:
# parameter settings

DEVICE = 'cuda:0'

DATA_NAME = 'PEMS08'

nlayer = 10
batch_size = 128

tid_count = 288
tid_tol = 3

torch.set_default_device(DEVICE)
data_type = torch.float32
gamma_list = [10 for _ in range(nlayer)]
beta_list = [1.5 for _ in range(nlayer)]


#### Prepare Data

Download the data following the instruction of [BasicTS](https://github.com/GestaltCogTeam/BasicTS/blob/master/tutorial/dataset_design.md) and unzip the data into `datasets/` directory.

In [ ]:
construct_data(DATA_NAME)

tr_inp = torch.from_numpy(np.load('datasets/constructed_data/train_inp_{}.npy'.format(DATA_NAME))).type(data_type).to('cpu')  # [B, L, N, C]
tr_tid = tr_inp[:, 0, 0, 1]  # [B]
tr_inp = tr_inp[:, :, :, 0]  # [B, L, N]

tr_tar = torch.from_numpy(np.load('datasets/constructed_data/train_tar_{}.npy'.format(DATA_NAME))).type(data_type).to('cpu')
tr_tar = tr_tar[:, :, :, 0]  # [B, L, N]

te_inp = torch.from_numpy(np.load('datasets/constructed_data/test_inp_{}.npy'.format(DATA_NAME))).type(data_type).to('cpu')
te_tid = te_inp[:, 0, 0, 1]  # [B]
te_inp = te_inp[:, :, :, 0]  # [B, L, N]


te_tar = torch.from_numpy(np.load('datasets/constructed_data/test_tar_{}.npy'.format(DATA_NAME))).type(data_type).to('cpu')
te_tar = te_tar[:, :, :, 0]  # [B, L, N]


### Prediction

In [4]:

pred = [torch.zeros_like(te_tar, dtype=te_tar.dtype).to(DEVICE) for _ in range(nlayer)]  # store the prediction results


for n in tqdm(range(tr_inp.shape[2]), desc='Predicting node:'):

    # bank construct

    tr_inp_seq = [torch.empty_like(tr_inp[:, :, n].to(DEVICE)) for _ in range(nlayer)]
    tr_del_seq = [torch.empty_like(tr_tar[:, :, n].to(DEVICE)) for _ in range(nlayer)]

    tr_inp_seq[0] = tr_inp[:, :, n].to(DEVICE)
    tr_del_seq[0] = tr_tar[:, :, n].to(DEVICE)

    L = tr_inp.shape[1]

    for i in range(1, nlayer):
        gamma = gamma_list[i-1]
        beta = beta_list[i-1]

        tr_inp_node = tr_inp_seq[i-1]  # [B, L]
        tr_del_node = tr_del_seq[i-1]  # [B, L]

        if i == 1:
            _, tr_inp_seq[i], tr_del_seq[i] = timewise_decouple_layer(tr_inp_node, tr_del_node, tr_inp_node, tr_del_node, 
                                                                      tid_dif=0, tid_count=tid_count, tid_tol=tid_tol, build_bank=True, 
                                                                      gamma=gamma, beta=beta)
        else:
            _, tr_inp_seq[i], tr_del_seq[i] = decouple_layer(tr_inp_node, tr_del_node, tr_inp_node, tr_del_node, True, batch_size, gamma, beta)
    
    
    # prediction
    
    te_inp_node = te_inp[:, :, n].to(DEVICE) + 0

    for i in range(nlayer):
        gamma = gamma_list[i]
        beta = beta_list[i]

        tr_inp_node = tr_inp_seq[i]
        tr_del_node = tr_del_seq[i]

        if i == 0:            
            pred[i][:, :, n], te_inp_node, _ = timewise_decouple_layer(tr_inp_node, tr_del_node, te_inp_node, 
                                                                       tid_dif=round(((te_tid[0] - tr_tid[0] +1)%1).item()*tid_count), 
                                                                       gamma=gamma, beta=beta)
        else:
            pred[i][:, :, n], te_inp_node, _ = decouple_layer(tr_inp_node, tr_del_node, te_inp_node, batch_size=batch_size, gamma=gamma, beta=beta)


Predicting node:: 100%|██████████| 170/170 [02:20<00:00,  1.21it/s]


#### Metrics

In [5]:
te_tar = te_tar.to(DEVICE)

# Main Performance

print('Overall MAE: %.3f' % masked_mae(torch.stack(pred).sum(dim=0), te_tar))
print('Overall RMSE: %.3f' % masked_rmse(torch.stack(pred).sum(dim=0), te_tar))
print('Overall MAPE: %.4f' % masked_mape(torch.stack(pred).sum(dim=0), te_tar))

print('Step 3:')
print('Overall MAE: %.3f' % masked_mae(torch.stack(pred).sum(dim=0)[:, 2, :], te_tar[:, 2, :]))
print('Overall RMSE: %.3f' % masked_rmse(torch.stack(pred).sum(dim=0)[:, 2, :], te_tar[:, 2, :]))
print('Overall MAPE: %.4f' % masked_mape(torch.stack(pred).sum(dim=0)[:, 2, :], te_tar[:, 2, :]))

print('Step 6:')
print('Overall MAE: %.3f' % masked_mae(torch.stack(pred).sum(dim=0)[:, 5, :], te_tar[:, 5, :]))
print('Overall RMSE: %.3f' % masked_rmse(torch.stack(pred).sum(dim=0)[:, 5, :], te_tar[:, 5, :]))
print('Overall MAPE: %.4f' % masked_mape(torch.stack(pred).sum(dim=0)[:, 5, :], te_tar[:, 5, :]))

print('Step 12:')
print('Overall MAE: %.3f' % masked_mae(torch.stack(pred).sum(dim=0)[:, 11, :], te_tar[:, 11, :]))
print('Overall RMSE: %.3f' % masked_rmse(torch.stack(pred).sum(dim=0)[:, 11, :], te_tar[:, 11, :]))
print('Overall MAPE: %.4f' % masked_mape(torch.stack(pred).sum(dim=0)[:, 11, :], te_tar[:, 11, :]))


Overall MAE: 14.579
Overall RMSE: 24.800
Overall MAPE: 0.0958
Step 3:
Overall MAE: 13.363
Overall RMSE: 22.594
Overall MAPE: 0.0870
Step 6:
Overall MAE: 14.496
Overall RMSE: 24.676
Overall MAPE: 0.0949
Step 12:
Overall MAE: 16.447
Overall RMSE: 27.833
Overall MAPE: 0.1099


In [6]:
# Performance after each layer

mae_temp = te_tar + 0
for i in range(nlayer):
        print('layer %d -> MAE: %.3f; RMSE: %.3f; MAPE: %.4f' % (i+1, masked_mae(torch.stack(pred)[:i+1].sum(dim=0), te_tar), masked_rmse(torch.stack(pred)[:i+1].sum(dim=0), te_tar), masked_mape(torch.stack(pred)[:i+1].sum(dim=0), te_tar)))
# print('Overall MAE: %.3f' % abs(te_tar - te_inp - torch.stack(pred).sum(dim=0)).mean().item())

layer 1 -> MAE: 16.961; RMSE: 27.336; MAPE: 0.1129
layer 2 -> MAE: 15.397; RMSE: 25.313; MAPE: 0.1002
layer 3 -> MAE: 14.996; RMSE: 25.083; MAPE: 0.0984
layer 4 -> MAE: 14.743; RMSE: 24.981; MAPE: 0.0968
layer 5 -> MAE: 14.673; RMSE: 24.924; MAPE: 0.0964
layer 6 -> MAE: 14.638; RMSE: 24.883; MAPE: 0.0962
layer 7 -> MAE: 14.616; RMSE: 24.854; MAPE: 0.0960
layer 8 -> MAE: 14.600; RMSE: 24.831; MAPE: 0.0960
layer 9 -> MAE: 14.589; RMSE: 24.813; MAPE: 0.0959
layer 10 -> MAE: 14.579; RMSE: 24.800; MAPE: 0.0958


In [7]:
torch.cuda.empty_cache()